In [8]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

In [9]:
np.random.seed(33550336)
rng = np.random.default_rng()

In [10]:
class Value:
    def __init__(self, data, children = (), op = ""):
        self.data = data
        self.grad = 0.0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None
    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        cool = self.data + other.data
        coolio = Value(cool, children = (self, other), op = "+")
        def _fackward():
            self.grad += coolio.grad
            other.grad += coolio.grad
        coolio._backward = _fackward
        return coolio
    def __radd__(self, other):
        return self.__add__(other)
    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        someswing = self.data * other.data
        something = Value(someswing, children = (self, other), op = "*")
        def _sackward():
            self.grad += other.data * something.grad
            other.grad += self.data * something.grad
        something._backward = _sackward
        return something
    def __rmul__(self, other):
        return self.__mul__(other)
    def backward(self):
        topo = []
        vis = set()
        def build_topo(node):
            if node in vis:
                return
            else:
                vis.add(node)
                parents = node._prev
                for parent in parents:
                    build_topo(parent)
                topo.append(node)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    def __pow__ (self, power):
        pikachu = self.data ** power
        pikajew = Value(pikachu, children=(self,), op="^")
        def _packward():
            self.grad += pikajew.grad * power * self.data ** (power - 1 )
        pikajew._backward = _packward
        return pikajew
    def relu(self):
        spin = max(0, self.data)
        bot = Value(spin, children = (self,), op = "ReLU")
        def _rackward():
            self.grad += bot.grad * (1 if self.data > 0 else 0)
        bot._backward = _rackward
        return bot

In [20]:
class Neuron:
    def __init__(self, nin, nonlin = True):
        self.ws = []
        self.nonlin = nonlin
        for i in range(nin):
            tmp = np.random.uniform(-1, 1)
            nocluewhatimdoing = Value(tmp)
            self.ws.append(nocluewhatimdoing)
        self.bias = Value(0)
    def __call__(self, x):
        igaveasecondchance = self.bias
        for w, xi in zip(self.ws, x):
            igaveasecondchance += w * xi
        if self.nonlin:
            atcupid = igaveasecondchance.relu()
        else:
            atcupid = igaveasecondchance
        return atcupid
    def parameters(self):
        return self.ws + [self.bias]

In [22]:
class Layer:
    def __init__(self, nin, nout, nonlin = True):
        self.neurons = []
        for _ in range(nout):
            self.neurons.append(Neuron(nin, nonlin = nonlin))
    def __call__(self, x):
        vas = []
        for neuron in self.neurons:
            thing = neuron(x)
            vas.append(thing)
        return vas
    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

In [23]:
class MLP:
    def __init__(self, nin, nouts, nonlin = True):
        sizes = [nin] + nouts
        self.layers = []
        for i in range(len(sizes) - 1):
            if i != len(sizes) - 2:
                self.layers.append(Layer(sizes[i], sizes[i + 1], nonlin = nonlin))
            else:
                self.layers.append(Layer(sizes[i], sizes[i + 1], nonlin = False))
    def __call__(self, x):
        for sm in self.layers:
            x = sm(x)
        return x
    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

In [13]:
layer = Layer(3, 4)
outputs = layer([1.0, 2.0, 3.0])

print(len(outputs))
print([isinstance(v, Value) for v in outputs])

4
[True, True, True, True]


In [14]:
a = Value(2)
b = Value(3)

c = a * b
L = c + a

L.backward()

print("Loss:", L.data)
print("a gradient:", a.grad)
print("b gradient:", b.grad)
print("c gradient:", c.grad)

Loss: 8
a gradient: 4.0
b gradient: 2.0
c gradient: 1.0


In [15]:
layer = Layer(3, 4)

print(len(layer.neurons))
print(len(layer.parameters()))

4
16


In [17]:
model = MLP(3, [4, 4, 1])
output = model([1.0, 2.0, 3.0])

print(len(output))
print(output[0].data)

1
0


In [19]:
model = MLP(3, [4, 4, 1])

print("Layers:", len(model.layers))
print("Parameters:", len(model.parameters()))

output = model([1.0, 2.0, 3.0])
print("Output:", output[0].data)

Layers: 3
Parameters: 41
Output: 0


In [24]:
model = MLP(3, [4, 4, 1])

print(len(model.parameters()))
print([layer.neurons[0].nonlin for layer in model.layers])

41
[True, True, False]


In [40]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [36]:
predictions = []

for x in xs:
    prediction = model(x)[0]
    predictions.append(prediction)

los = 0
for pred, targ in zip(predictions, ys):
    los += (targ + (-1) * pred) ** 2

los *= (1/len(xs))

In [59]:
model = MLP(3, [4, 4, 1])
eta = 0.01
for _ in range(300):
    predictions = []
    for x in xs:
        prediction = model(x)[0]
        predictions.append(prediction)
    los = 0
    for pred, targ in zip(predictions, ys):
        los += (targ + (-1) * pred) ** 2
    los *= (1/len(xs))
    if _ == 0:
        print(los.data)
    for param in model.parameters():
        param.grad = 0
    los.backward()
    for param in model.parameters():
        param.data = param.data - eta * param.grad
    print(los.data)

2.334066691722075
2.334066691722075
1.9570003511147465
1.6959481867702313
1.5044054977738595
1.3576473470182857
1.2410927823919877
1.1455181610642153
1.0659047915892081
0.9984448415354927
0.9404607323287915
0.8900176699639448
0.8456844673893595
0.819981227076612
0.8106801267603314
0.8017425694956479
0.793144692107211
0.784864556929979
0.7768819685821754
0.7691783109040093
0.7617364015433685
0.7545403620230451
0.7475755014220143
0.7408282120554677
0.7342858757525305
0.7279367795132133
0.7217700394823248
0.7157755323120598
0.709943833100267
0.704266159190865
0.6987343192089321
0.6933406667776134
0.6880780584288564
0.6829398152764967
0.677919688069537
0.6730118252866144
0.6692214027542813
0.6671978939894196
0.6651875231633921
0.6631895905507988
0.6612034660771356
0.6592285783009394
0.6572644053502712
0.6553104674636733
0.6533663208472008
0.651431552610148
0.6495057765843424
0.6475886298668228
0.645679769954545
0.6437788723635188
0.6418856286443315
0.639999744722063
0.6381209395017824
0.63

In [46]:
for a, b in zip(xs, ys):
    print(model(a)[0].data)
    print(b)

1.03857034174604
1.0
-0.9110966136241171
-1.0
-0.9110966136241171
-1.0
0.9221039164353456
1.0


In [65]:
los = 0
predictions = []
for x in xs:
    prediction = model(x)[0]
    predictions.append(prediction)
for pred, targ in zip(predictions, ys):
    los += (targ + (-1) * pred) ** 2
los *- (1/len(xs))
for param in model.parameters():
    param.grad = 0    
los.backward()
print(model.parameters()[40].grad)

-0.6119892203332387


In [76]:
param = model.parameters()[40]
pd = param.data
epsilon = 1e-5
param.data += 1e-5
los = 0
predictions = []
for x in xs:
    prediction = model(x)[0]
    predictions.append(prediction)
for pred, targ in zip(predictions, ys):
    los += (targ + (-1) * pred) ** 2
los *= (1/len(xs))
lplus = los
param.data = pd
param.data -= 1e-5
los = 0
predictions = []
for x in xs:
    prediction = model(x)[0]
    predictions.append(prediction)
for pred, targ in zip(predictions, ys):
    los += (targ + (-1) * pred) ** 2
los *= (1/len(xs))
lminus = los
param.data = pd
print((lplus.data + ((-1) * lminus.data)) * 1/(2 * epsilon))

-0.1530773050817391


In [79]:
predictions = []
for x in xs:
    prediction = model(x)[0]
    predictions.append(prediction)

los = 0
for pred, targ in zip(predictions, ys):
    los += (targ + (-1) * pred) ** 2

los *= (1/(len(xs)))
for param in model.parameters():
    param.grad = 0 

los.backward()
print(model.parameters()[40].grad)

-0.1530773050833093
